# Quality control 

In [2]:
# import scvi
import scanpy as sc

import scipy

import celltypist
from celltypist import models

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import sys
import os

ModuleNotFoundError: No module named 'celltypist'

In [3]:
sc.__version__

'1.11.3'

# Reset logging configuration to default

In [ ]:
import logging
logging.getLogger().handlers = []
logging.basicConfig(level=logging.WARNING)

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

## Custom modules

In [ ]:
sys.path.append('bin/')
from adata_qc import *

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.4.1-FD20250213HSCT/lib/R'

In [ ]:
import rpy2
%load_ext rpy2.ipython

## Figures 

In [ ]:
sc.set_figure_params(figsize=(7, 7), dpi_save=1200, fontsize=12, frameon=False, facecolor='white')
mpl.rcParams['figure.facecolor'] = 'white'

# Files and parameter 

In [ ]:
# Seurat object 
raw_h5ad_file = 'data/object/raw.h5ad'
qc_h5ad_file = 'data/object/qc.h5ad'

In [ ]:
# QC matrix 
nUMI_min = 800
nFeature_min = 600
pMT_max = 10
hto_demux_class = ['Blood', 'Skin']

# Import filtered CellRanger h5ad

In [ ]:
adata = sc.read_h5ad(raw_h5ad_file)
adata.raw = adata.copy()

# Compute QC matrix

In [ ]:
adata.var['MT'] = adata.var_names.str.startswith('MT-')
adata.var['HB'] = adata.var_names.str.startswith('HBA', 'HBB')
adata.var['RP'] = adata.var_names.str.startswith('RPL', 'RPS')

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['MT', 'HB', 'RP'], percent_top=None, inplace=True)

# LogNormalize data 

In [ ]:
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)

# Compute cell cycle score 

In [ ]:
%%R -o s_genes -o g2m_genes
s_genes = Seurat::cc.genes.updated.2019$s.genes
g2m_genes = Seurat::cc.genes.updated.2019$g2m.genes

In [ ]:
sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
adata.obs.rename(columns={'phase': 'msCC_label'}, inplace=True)

# Compute celltypist cell type labels

In [ ]:
# CellTypist models
models.download_models(force_update=False)

In [ ]:
# Predict cell types (Immune_All_High.pkl)
adata = celltypist.annotate(adata, model='Immune_All_High.pkl', majority_voting=False).to_adata()

# Set result names
adata.obs.rename(columns={'predicted_labels': 'celltypist_label_high', 'conf_score':'celltypist_score_high'}, inplace=True)

In [ ]:
# Predict cell types (Immune_All_Low.pkl)
predictions=celltypist.annotate(adata, model='Immune_All_Low.pkl', majority_voting=False).to_adata()

# Set result names
adata.obs.rename(columns={'predicted_labels': 'celltypist_label_low', 'conf_score':'celltypist_score_low'}, inplace=True)

# Quality control plots

In [ ]:
plot_qc_density(adata, 'total_counts', cutoff=nUMI_min, n_cols=8)

In [ ]:
plot_qc_density(adata, 'n_genes_by_counts', cutoff=nFeature_min, n_cols=8)

In [ ]:
plot_qc_density(adata, 'pct_counts_MT', cutoff=pMT_max, n_cols=8)

In [ ]:
plot_qc_density(adata, 'pct_counts_RP', cutoff=None, n_cols=8)

In [ ]:
plot_qc_scatter(adata, 'total_counts', 'n_genes_by_counts', x_cutoff=nUMI_min, y_cutoff=nFeature_min, sample_col='sample_name', dot_size=20, n_cols=8)

# Set raw count matrix

In [ ]:
adata = adata.raw.to_adata()
adata.raw = adata.copy()

# Filter by QC matrix 

In [ ]:
adata = adata[

    (adata.obs['total_counts'] >= nUMI_min) & \
    (adata.obs['n_genes_by_counts'] >= nFeature_min) & \
    (adata.obs['pct_counts_MT'] <= pMT_max)
    
]

# Solo detection on facility subsets

## Train SCVI model

In [ ]:
cache_scvi = True

In [ ]:
def model_history(model): 
    
    # Plot model history 
    fig, axes=plt.subplots(1, 2, figsize=(10, 5))
    
    axes[0].plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
    axes[0].plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')
    axes[0].set_title('Reconstruction Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    
    axes[1].plot(model.history['elbo_train']['elbo_train'], label='train')
    axes[1].plot(model.history['elbo_validation']['elbo_validation'], label='validation')
    axes[1].set_title('ELBO')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

In [ ]:
def scvi_workflow(adata, subset=None): 

    # Subset genes
    adata=adata[adata.obs.sample_name==subset]
    adata=adata[:, (adata.X>=1).sum(axis=0)>=1]

    # Ensure copy
    adata=adata.copy()

    # Setup anndata
    scvi.model.SCVI.setup_anndata(
    
        adata, 
        batch_key='sample_name'

    )
    
    # Model 
    model=scvi.model.SCVI(
    
        adata, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    # Max epochs heuristic 
    max_epochs=int(np.min([round((20000 / adata.n_obs) * 400), 400]))
    
    # Train model 
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)

    # Model history
    model_history(model)
    
    # Save results 
    model.save('data/object/solo/model_scvi_' + subset, overwrite=True, save_anndata=False)
    adata.write_h5ad('data/object/solo/adata_' + subset + '.h5ad')

In [ ]:
if not cache_scvi: 

    # Run SCVI
    for subset in adata.obs['sample_name'].cat.categories: 
        scvi_workflow(adata.copy(), subset)
        
else: 
    
    for subset in adata.obs['sample_name'].cat.categories: 
        
        # Plot model history
        # model_history(scvi.model.SCVI.load('data/object/solo/model_scvi_' + subset, adata=sc.read_h5ad('data/object/solo/adata_' + subset + '.h5ad')))     

        None

## Solo doublet detection 

In [ ]:
cache_solo = True

In [ ]:
def solo_workflow(subset=None): 

    # Import SCVI model 
    adata=sc.read_h5ad('data/object/solo/adata_' + subset + '.h5ad')
    model=scvi.model.SCVI.load('data/object/solo/model_scvi_' + subset, adata=adata)

    # Register solo model 
    model_solo=scvi.external.SOLO.from_scvi_model(model)
    
    # Max epochs heuristic 
    max_epochs=int(np.min([round((20000 / adata.n_obs) * 400), 400]))
    
    # Train solo
    model_solo.train(max_epochs=max_epochs, check_val_every_n_epoch=1)

    # Get doublet score and labels 
    prob=model_solo.predict(soft=True, include_simulated_doublets=False)
    label=model_solo.predict(soft=False, include_simulated_doublets=False)

    adata.obs['solo_pred_doublet']=prob['doublet']
    adata.obs['solo_pred_singlet']=prob['singlet']
    adata.obs['solo_label']=label

    # Store results 
    model_solo.save('data/object/solo/model_solo_' + subset, overwrite=True, save_anndata=False)
    adata.write_h5ad('data/object/solo/adata_' + subset + '.h5ad')

In [ ]:
if not cache_solo: 

    # Run SOlO
    for subset in adata.obs['sample_name'].cat.categories: 
        solo_workflow(subset)

## Fetch SOLO result 

In [ ]:
solo = [sc.read_h5ad('data/object/solo/adata_' + subset + '.h5ad').obs[['solo_pred_singlet', 'solo_pred_doublet', 'solo_label']] for subset in adata.obs['sample_name'].cat.categories]
solo = pd.concat(solo)

In [ ]:
adata.obs = adata.obs.merge(solo, left_index=True, right_index=True, how='left')

# Train SCVI model

In [ ]:
adata = adata[:, (adata.X>=3).sum(axis=0)>=3].copy()

In [ ]:
cache_scvi = True

In [ ]:
if not cache_scvi: 
    
    # Setup anndata
    scvi.model.SCVI.setup_anndata(
    
        adata,
        batch_key='sample_name'

    )
    
    # Model 
    model = scvi.model.SCVI(
    
        adata, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    # Max epochs heuristic 
    max_epochs = int(np.min([round((20000 / adata.n_obs) * 400), 400]))
    
    # Train model 
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    # Save results 
    model.save('data/object/scvi/qc/model/', overwrite=True)
    
else: 
    
    model = scvi.model.SCVI.load('data/object/scvi/qc/model/', adata=adata)

# Training validation 

In [ ]:
# Plot model history 
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].plot(model.history['reconstruction_loss_train']['reconstruction_loss_train'], label='train')
axes[0].plot(model.history['reconstruction_loss_validation']['reconstruction_loss_validation'], label='validation')
axes[0].set_title('Reconstruction Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(model.history['elbo_train']['elbo_train'], label='train')
axes[1].plot(model.history['elbo_validation']['elbo_validation'], label='validation')
axes[1].set_title('ELBO')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

# Dim reduction on latent space

In [ ]:
cache_dim = True

In [ ]:
if not cache_dim: 
    
    adata.obsm['latent'] = model.get_latent_representation()
    
    sc.pp.neighbors(adata, n_neighbors=30, use_rep='latent')
    sc.tl.leiden(adata, resolution=1, flavor='igraph', n_iterations=2, key_added='leiden')
    sc.tl.umap(adata, min_dist=1)
    
    adata.write_h5ad('data/object/scvi/qc/adata.h5ad')
    
else: 
    
    adata = sc.read_h5ad('data/object/scvi/qc/adata.h5ad')

# UMAP

In [ ]:
sc.pl.umap(adata, color=['leiden', 'patient_id', 'time_point', 'digestion_protocol', 'msCC_label', 'hto_demux_class', 'solo_label', 'log1p_total_counts', 'log1p_n_genes_by_counts', 'pct_counts_MT', 'pct_counts_RP', 'pct_counts_HB'], frameon=False, ncols=4, wspace=0.5, size=20)

# Set raw count matrix

In [ ]:
adata = adata.raw.to_adata()
adata.raw = adata.copy()

# Save results 

In [ ]:
adata.write_h5ad(qc_h5ad_file)